# Lecture 4 Colab: Imbalanced Fraud Detection

This notebook demonstrates why accuracy can be misleading in highly imbalanced fraud detection.


## Learning objectives

- Understand class imbalance.
- See why high accuracy can be misleading.
- Evaluate fraud detection using precision, recall, F1-score, ROC-AUC, and PR-AUC.
- Explore class weights and threshold tuning.
- Connect false positives and false negatives to fraud business costs.


## Connection to Lecture 4

Supports imbalanced data, accuracy, precision, recall, F1-score, ROC-AUC, PR-AUC, and business error costs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve
np.random.seed(42)


## Dataset explanation

We generate a synthetic fraud dataset with a rare positive class. The data is not real transaction data.


In [ ]:
feature_names = ["transaction_amount_score", "account_age_score", "merchant_risk_score", "device_change_score", "location_risk_score", "transaction_velocity", "failed_login_score", "historical_chargeback_score", "time_of_day_score", "digital_behavior_score"]
X, y = make_classification(n_samples=10000, n_features=10, n_informative=5, n_redundant=2, weights=[0.99, 0.01], class_sep=1.2, random_state=42)
df = pd.DataFrame(X, columns=feature_names)
df["fraud"] = y
df.head()


## Show class imbalance

Fraud is the positive class and is rare.


In [ ]:
counts = df["fraud"].value_counts().sort_index()
percentages = df["fraud"].value_counts(normalize=True).sort_index() * 100
print(pd.DataFrame({"count": counts, "percentage": percentages.round(2)}))
counts.plot(kind="bar", color=["steelblue", "indianred"])
plt.xticks([0, 1], ["Non-fraud", "Fraud"], rotation=0)
plt.ylabel("Count")
plt.title("Fraud class imbalance")
plt.show()


## Baseline accuracy trap

A model that predicts all transactions as non-fraud can have very high accuracy but zero fraud recall.


In [ ]:
y_dummy = np.zeros_like(y)
print("Accuracy :", round(accuracy_score(y, y_dummy), 3))
print("Precision:", round(precision_score(y, y_dummy, zero_division=0), 3))
print("Recall   :", round(recall_score(y, y_dummy, zero_division=0), 3))
print("F1-score :", round(f1_score(y, y_dummy, zero_division=0), 3))


## Train/test split

Use stratification so both sets contain fraud examples.


In [ ]:
X = df[feature_names]
y = df["fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Training fraud rate:", round(y_train.mean(), 4))
print("Test fraud rate:", round(y_test.mean(), 4))


## Train models

Class weights tell the model to pay more attention to the rare fraud class.


In [ ]:
normal_log_reg = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
balanced_log_reg = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
rf_balanced = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
normal_log_reg.fit(X_train, y_train)
balanced_log_reg.fit(X_train, y_train)
rf_balanced.fit(X_train, y_train)


## Evaluation helper

The threshold converts model scores into fraud alerts.


In [ ]:
def evaluate_binary_model(name, model, X_test, y_test, threshold=0.5):
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)
    print(f"{name} at threshold {threshold}")
    print("Accuracy :", round(accuracy_score(y_test, preds), 3))
    print("Precision:", round(precision_score(y_test, preds, zero_division=0), 3))
    print("Recall   :", round(recall_score(y_test, preds, zero_division=0), 3))
    print("F1-score :", round(f1_score(y_test, preds, zero_division=0), 3))
    print("ROC-AUC  :", round(roc_auc_score(y_test, probs), 3))
    print("PR-AUC   :", round(average_precision_score(y_test, probs), 3))
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=["Non-fraud", "Fraud"]).plot(cmap="Blues", values_format="d")
    plt.title(name)
    plt.show()
    return probs, preds


## Evaluate models

Compare normal logistic regression, balanced logistic regression, and balanced random forest.


In [ ]:
normal_probs, _ = evaluate_binary_model("Normal logistic regression", normal_log_reg, X_test, y_test)
balanced_probs, _ = evaluate_binary_model("Balanced logistic regression", balanced_log_reg, X_test, y_test)
rf_probs, _ = evaluate_binary_model("Balanced random forest", rf_balanced, X_test, y_test)


## ROC and PR curves

PR curves are often more informative when the positive class is rare.


In [ ]:
models = [("Normal LR", normal_probs), ("Balanced LR", balanced_probs), ("Balanced RF", rf_probs)]
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
for name, probs in models:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} AUC={roc_auc_score(y_test, probs):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves")
plt.legend()
plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
for name, probs in models:
    precision, recall, _ = precision_recall_curve(y_test, probs)
    plt.plot(recall, precision, label=f"{name} PR-AUC={average_precision_score(y_test, probs):.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Threshold tuning

The default 0.5 threshold is not automatically correct.


In [ ]:
rows = []
for threshold in [0.1, 0.2, 0.3, 0.5, 0.7]:
    preds = (balanced_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    rows.append({"threshold": threshold, "precision": precision_score(y_test, preds, zero_division=0), "recall": recall_score(y_test, preds, zero_division=0), "f1": f1_score(y_test, preds, zero_division=0), "false_positives": fp, "false_negatives": fn})
pd.DataFrame(rows).round(3)


## Business interpretation

- False positive: legitimate transaction blocked or sent to manual review.
- False negative: fraudulent transaction missed.

Recall may be important, but too many false positives can hurt customer experience and overload analysts.


## What to observe

- High accuracy can miss all fraud.
- Class weights change model behaviour.
- Threshold tuning connects model score to operational decisions.


## Common pitfalls

- Reporting only accuracy.
- Oversampling before train/test split.
- Ignoring business review capacity.
- Assuming the default 0.5 threshold is always correct.
- Using ROC-AUC without checking precision-recall trade-off.


## Try it yourself

- Change fraud rate from 1% to 5%.
- Change threshold.
- Compare `class_weight=None` vs `class_weight='balanced'`.
- Decide a threshold for a bank that wants high recall but limited false alerts.


## Reflection questions

- Why is 99% accuracy misleading?
- Which metric best reflects missed fraud?
- Which metric best reflects false alerts?
- Why is PR-AUC useful for rare events?


## Final summary

Imbalanced data requires careful metrics. Accuracy alone can be dangerous. Threshold tuning connects model score to operational decisions.
